# E4 — Evaluation: LLM vs ML control
Metric suite from the plan (§4), scored **only against real ExtraSensory self-report labels**:
per-field **macro-F1 + balanced accuracy**, per-class F1, coverage, over-confidence, confusion
matrices, LLM↔ML agreement (Cohen's κ), and a **bootstrap CI by user** on the macro-F1 gap.

- **ML** is reported on the full test set (each user tested once across the 5 folds).
- **Head-to-head** (ML vs LLM) is on the shared stratified eval sample from E3.
- LLM sections activate automatically once `e3_pred_llm.parquet` exists.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import (f1_score, balanced_accuracy_score,
                             classification_report, confusion_matrix, cohen_kappa_score)
import ee_common as ee
ee.ensure_output_dirs()

key = ["uuid", "timestamp"]
gold = pd.read_parquet(ee.DATA_DIR / "e2_gold_labels.parquet").set_index(key).sort_index()
ml = pd.read_parquet(ee.RESULTS_DIR / "e3_pred_ml.parquet").set_index(key).sort_index()
eval_s = pd.read_parquet(ee.RESULTS_DIR / "e3_eval_sample.parquet").set_index(key).sort_index()
llm_path = ee.RESULTS_DIR / "e3_pred_llm.parquet"
has_llm = llm_path.exists()
llm = pd.read_parquet(llm_path).set_index(key).sort_index() if has_llm else None
print("ML preds:", len(ml), "| eval sample:", len(eval_s), "| LLM preds:", (len(llm) if has_llm else "—"))

ML preds: 377346 | eval sample: 1530 | LLM preds: 1530


## 1. Metric helpers (scored on eligible samples only)

In [2]:
def field_scores(pred, idx, field):
    """macro-F1 + balanced accuracy for one field over eligible rows in idx."""
    sub = gold.loc[idx]
    elig = sub[f"elig_{field}"].values
    y = sub.loc[elig, f"gold_{field}"]
    yhat = pred.loc[y.index, f"pred_{field}"]
    return {"macro_f1": f1_score(y, yhat, average="macro", zero_division=0),
            "balanced_acc": balanced_accuracy_score(y, yhat),
            "n": int(elig.sum())}

def coverage_overconfidence(pred, idx, field, none_label):
    """coverage = share of committed (non-none) preds; over-confidence = committing when gold==none."""
    sub = gold.loc[idx]; elig = sub[f"elig_{field}"].values
    y = sub.loc[elig, f"gold_{field}"]; yhat = pred.loc[y.index, f"pred_{field}"]
    committed = yhat != none_label
    gold_none = y == none_label
    acc_committed = (yhat[committed & ~gold_none.values] ==
                     y[committed & ~gold_none.values]).mean() if committed.any() else np.nan
    overconf = (yhat[gold_none.values] != none_label).mean() if gold_none.any() else np.nan
    return {"coverage": committed.mean(), "acc_on_committed": acc_committed,
            "over_confidence": overconf}

## 2. ML on the full test set

In [3]:
rows = []
for f in ee.FIELDS:
    s = field_scores(ml, ml.index, f)
    rows.append({"field": f, **s})
ml_full = pd.DataFrame(rows).set_index("field")
ml_full.to_csv(ee.RESULTS_DIR / "e4_ml_full_scores.csv")
ml_full

,macro_f1,balanced_acc,n
field,,,
location,0.291233,0.342008,372168
activity,0.293157,0.326934,308320
companion,0.422833,0.451698,192771


## 3. Head-to-head on the shared eval sample

In [4]:
none_of = {"location": "unknown", "activity": "unknown", "companion": "alone"}
enrichers = {"ML": ml}
if has_llm:
    enrichers["LLM"] = llm

comp = []
for name, pred in enrichers.items():
    for f in ee.FIELDS:
        s = field_scores(pred, eval_s.index, f)
        c = coverage_overconfidence(pred, eval_s.index, f, none_of[f])
        comp.append({"enricher": name, "field": f, **s, **c})
comparison = pd.DataFrame(comp)
comparison.to_csv(ee.RESULTS_DIR / "e4_comparison.csv", index=False)
comparison.round(3)

,enricher,field,macro_f1,balanced_acc,n,coverage,acc_on_committed,over_confidence
0,ML,location,0.344,0.358,1522,0.812,0.470,0.771
1,ML,activity,0.283,0.317,1380,0.949,0.412,0.860
2,ML,companion,0.403,0.421,802,0.390,0.805,0.339
3,LLM,location,0.248,0.291,1522,0.850,0.401,0.911
4,LLM,activity,0.229,0.253,1380,0.941,0.426,0.920
5,LLM,companion,0.287,0.335,802,0.011,1.000,0.010


## 4. Bootstrap CI (by user) on the macro-F1 gap  LLM − ML

In [5]:
if has_llm:
    users = eval_s.index.get_level_values("uuid").unique().to_numpy()
    rng = np.random.default_rng(0)
    def macro_f1_gap(field, uu):
        idx = eval_s.index[eval_s.index.get_level_values("uuid").isin(uu)]
        return (field_scores(llm, idx, field)["macro_f1"]
                - field_scores(ml, idx, field)["macro_f1"])
    for f in ee.FIELDS:
        boots = [macro_f1_gap(f, rng.choice(users, len(users), replace=True)) for _ in range(1000)]
        lo, hi = np.percentile(boots, [2.5, 97.5])
        point = macro_f1_gap(f, users)
        verdict = "LLM wins" if lo > 0 else ("ML wins" if hi < 0 else "tie")
        print(f"[{f:9s}] LLM−ML macro-F1 = {point:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}]  -> {verdict}")
else:
    print("LLM predictions not present yet — add the key, run E3 cell 5, then re-run E4.")

[location ] LLM−ML macro-F1 = -0.096  95% CI [-0.133, -0.039]  -> ML wins


C:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
C:\Users\User\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:2801: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


[activity ] LLM−ML macro-F1 = -0.055  95% CI [-0.104, -0.001]  -> ML wins


[companion] LLM−ML macro-F1 = -0.116  95% CI [-0.161, -0.049]  -> ML wins


## 5. LLM cost / latency

In [6]:
import json
cost_path = ee.RESULTS_DIR / "e3_llm_cost.json"
if cost_path.exists():
    cost = json.load(open(cost_path))
    print("LLM cost/latency (from E3):")
    for k, v in cost.items():
        print(f"  {k}: {v}")
    print(f"  ~seconds/sample: {cost.get('mean_latency_s', 0):.2f}")
else:
    print("No e3_llm_cost.json yet — run E3 cell 5 with a key first.")

LLM cost/latency (from E3):
  n_eval: 1530
  n_live_calls: 1503
  mean_latency_s: 0.842623231415739
  parse_errors: 0
  model: gemini-3.1-flash-lite
  mode: few-shot
  ~seconds/sample: 0.84


## 6. Confusion matrices + LLM↔ML agreement

In [7]:
for f in ee.FIELDS:
    sub = gold.loc[eval_s.index]; elig = sub[f"elig_{f}"].values
    y = sub.loc[elig, f"gold_{f}"]
    labels = sorted(y.unique())
    print(f"=== {f} confusion (rows=gold, cols=ML pred) ===")
    cm = confusion_matrix(y, ml.loc[y.index, f"pred_{f}"], labels=labels)
    print(pd.DataFrame(cm, index=labels, columns=labels).to_string())
    if has_llm:
        k = cohen_kappa_score(ml.loc[y.index, f"pred_{f}"], llm.loc[y.index, f"pred_{f}"])
        print(f"LLM↔ML Cohen's kappa [{f}]: {k:.3f}")
    print()

=== location confusion (rows=gold, cols=ML pred) ===
            beach  gym  home  outdoors  restaurant  transit  unknown  work
beach          29   27    19        23           3        9       20     0
gym            28    5    33        23          10        5       40    56
home            0    0   127         6           2        1       49    15
outdoors        5   16    27        48          10       25       49    20
restaurant      2    9    23        15          69       31       38    13
transit         3   10    22        14           8      106       22    15
unknown         3    2    82        13           8        9       44    31
work            0    5    29         7           5        3       24   127
LLM↔ML Cohen's kappa [location]: 0.218

=== activity confusion (rows=gold, cols=ML pred) ===
          cycling  eating  exercise  running  sitting  sleeping  standing  unknown  walking
cycling        14       5         0        2        2         0         3        1     

## 7. Transformer arms — multi-arm comparison (Task 3 / RQ1c)
Self-built Transformer variants from `E6_transformer.py`, added as extra arms beside ML + LLM
and scored the same way (macro-F1 over eligible rows; full test + shared eval sample):

- **TF-feature** (A) — FT-Transformer, attention over the 225 features.
- **TF-group** (B) — attention over ~12 sensor-group tokens.
- **TF-temporal** (C) — attention over time (recent-sample window per user).

Each arm activates automatically once its `e6_pred_<variant>.parquet` exists — this section is a
no-op until E6 has run. Cell 1: multi-arm table (`e4_multi_arm.csv`). Cell 2: pairwise bootstrap
CI by user — each Transformer vs ML, vs LLM, and Transformer-vs-Transformer (`e4_multi_arm_ci.csv`).

In [ ]:
tf_variants = ["feature", "group", "temporal"]
tf_arms = {}
for v in tf_variants:
    p = ee.RESULTS_DIR / f"e6_pred_{v}.parquet"
    if p.exists():
        tf_arms[f"TF-{v}"] = pd.read_parquet(p).set_index(key).sort_index()
print("Transformer arms found:", list(tf_arms) or "none (run E6_transformer.py first)")

multi_arm = None
if tf_arms:
    arms = {"ML": ml, **({"LLM": llm} if has_llm else {}), **tf_arms}
    rows = []
    for name, pred in arms.items():
        # full-test macro-F1 only for arms covering the whole test set (ML + Transformers;
        # the LLM ran on the eval sample only). Every arm gets an eval-sample score.
        full_ok = pred.index.intersection(ml.index).shape[0] == ml.shape[0]
        for f in ee.FIELDS:
            ev = field_scores(pred, eval_s.index, f)
            c = coverage_overconfidence(pred, eval_s.index, f, none_of[f])
            rows.append({"arm": name, "field": f,
                         "macro_f1_full": round(field_scores(pred, ml.index, f)["macro_f1"], 4)
                                          if full_ok else np.nan,
                         "macro_f1_eval": round(ev["macro_f1"], 4),
                         "balanced_acc_eval": round(ev["balanced_acc"], 4),
                         "coverage": round(c["coverage"], 4),
                         "over_confidence": round(c["over_confidence"], 4)})
    multi_arm = pd.DataFrame(rows)
    multi_arm.to_csv(ee.RESULTS_DIR / "e4_multi_arm.csv", index=False)
    print("saved e4_multi_arm.csv — macro-F1 (eval sample) by arm x field:")
    print(multi_arm.pivot(index="arm", columns="field", values="macro_f1_eval").to_string())
multi_arm if multi_arm is not None else "run E6_transformer.py first"


In [ ]:
if tf_arms:
    arms_ci = {"ML": ml, **({"LLM": llm} if has_llm else {}), **tf_arms}
    users = eval_s.index.get_level_values("uuid").unique().to_numpy()
    rng = np.random.default_rng(0)

    def gap(a, b, field, uu):
        """macro-F1(a) - macro-F1(b) for one field over the eval rows of users `uu`."""
        idx = eval_s.index[eval_s.index.get_level_values("uuid").isin(uu)]
        return (field_scores(arms_ci[a], idx, field)["macro_f1"]
                - field_scores(arms_ci[b], idx, field)["macro_f1"])

    tf_names = list(tf_arms)
    pairs = [(t, "ML") for t in tf_names]                       # each Transformer vs ML
    if has_llm:
        pairs += [(t, "LLM") for t in tf_names]                 # each Transformer vs LLM
    pairs += [(tf_names[i], tf_names[j])                        # Transformer vs Transformer
              for i in range(len(tf_names)) for j in range(i + 1, len(tf_names))]

    ci_rows = []
    for a, b in pairs:
        for f in ee.FIELDS:
            boots = [gap(a, b, f, rng.choice(users, len(users), replace=True)) for _ in range(1000)]
            lo, hi = np.percentile(boots, [2.5, 97.5])
            point = gap(a, b, f, users)
            verdict = f"{a} wins" if lo > 0 else (f"{b} wins" if hi < 0 else "tie")
            ci_rows.append({"pair": f"{a}-{b}", "field": f, "gap": round(point, 4),
                            "ci_lo": round(lo, 4), "ci_hi": round(hi, 4), "verdict": verdict})
            print(f"[{f:9s}] {a}-{b} macro-F1 = {point:+.3f}  95% CI [{lo:+.3f}, {hi:+.3f}] -> {verdict}")
    ci = pd.DataFrame(ci_rows)
    ci.to_csv(ee.RESULTS_DIR / "e4_multi_arm_ci.csv", index=False)
    print("\nsaved e4_multi_arm_ci.csv")
else:
    print("No Transformer arms yet — run E6_transformer.py, then re-run E4.")


**E4 done.** Comparison table + per-field confusion saved to `results/enrichment_experiment/`
(`e4_comparison.csv`, `e4_ml_full_scores.csv`). Once the LLM has run, this notebook fills in the
LLM column, the bootstrap-CI verdict per field, over-confidence, LLM↔ML agreement, and the
cost/latency line — the final answer to RQ1 ("which enriches context better").